# Efficiency in Practice I: Introduction to Profiling with Python
**Memory Profiling**: Monitor CPU and RAM of your Python code.

#### Bytes-to-Human Helper

In [2]:
# Convert bytes to a human-readable format
def bytes_to_human(n):
    for unit in ["B", "KiB", "MiB", "GiB", "TiB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024

##### Head-start Example with Top to Down Profiling

In [5]:
import psutil

# System-wide resource usage (RAM stats)
# Have the options: total, available, used, percent, etc.
# Note: This is just a snapshot of the current system state, not specific to our process.
# Use it when asking: "How much RAM does the computer have/free right now?"
print("System-wide memory total:", bytes_to_human(psutil.virtual_memory().total))

# File system storage stats (disk)
# You can check total, used, free, and percent for a specific disk partition (e.g., "/").
# Note: This is about the disk storage, not RAM. It tells you how much space is used/free on your hard drive.
# Use it when asking: "How full is my disk?"
print("Disk usage:", bytes_to_human(psutil.disk_usage("/").used))

# Process-specific resource usage (RAM stats)
# Use it when asking: "How much RAM is my Python process using right now?"
print("Process memory usage:", bytes_to_human(psutil.Process().memory_info().rss))

System-wide memory total: 32.00 GiB
Disk usage: 11.60 GiB
Process memory usage: 85.20 MiB


## 1) Quick Snapshot with `psutil` (CPU + RAM)

Measures process-level memory from the OS (e.g., RSS - *resident set size*).
- Includes almost everything your Python process is using: Python objects, C extensions, native buffers, interpreter overhead, etc.
- **Best for**: “How much RAM is my program using overall?”

Use `psutil` for real-world RAM footprint.

In [6]:
import psutil
import os

# Creates a handle to one specific process, here your current Python process
process = psutil.Process(os.getpid())

# Memory details of that chosen process
mem = process.memory_info().rss  # RAM usage, resident set size in bytes
print("RAM:", bytes_to_human(mem))

# CPU usage (percentage over an interval)
print(f"CPU: {process.cpu_percent(interval=1):.2f}%")

RAM: 85.20 MiB
CPU: 0.40%


## 2) Track Peak RAM with `tracemalloc` (stdlib)

Measures Python allocation tracking (memory allocated by Python code).
- Great for comparing snapshots and identifying where Python-level allocations came from (line/file).
- Does not fully reflect all native/C-level allocations.
- **Best for**: “Which Python lines/objects are growing memory?”

Use `tracemalloc` for debugging memory growth in Python code.

In [7]:
import tracemalloc

tracemalloc.start()

# ... your code here ...

current, peak = tracemalloc.get_traced_memory()
print("Current memory usage:", tracemalloc._format_size(current, False))

# Note: tracemalloc's _format_size is an internal method, not part of the public API.
# Below are two alternatives to convert bytes to a human-readable format.

print("Peak memory usage:", tracemalloc._format_size(peak, False))
print("Peak memory usage:", bytes_to_human(peak))
tracemalloc.stop()

Current memory usage: 922 B
Peak memory usage: 20.4 KiB
Peak memory usage: 20.37 KiB


### ⭐ A wrapper for you to use later on ⬇️

In [ ]:
import tracemalloc


def analyze_func(func, *args):
    print(
        "..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::"
    )
    tracemalloc.start()
    func(*args)
    current, peak = tracemalloc.get_traced_memory()
    print("Current memory usage:", tracemalloc._format_size(current, False))
    print("Peak memory usage:", tracemalloc._format_size(peak, False))
    tracemalloc.stop()
    print(
        "..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::..::"
    )

## 3) Monitoring Continuously (wrap your workload)

Below are two examples on how to monitor your code continuously by using wrappers.

**Note:** `tracemalloc` tracks Python allocations, while `psutil` RSS reflects process memory from both Python and native extensions.

### 3.1) Continuous Monitoring Example: `psutil` + `threading`

In [8]:
import psutil, os, threading, time


def monitor(interval=0.5):
    proc = psutil.Process(os.getpid())
    while not stop_event.is_set():
        mem = proc.memory_info().rss
        cpu = proc.cpu_percent(interval=interval)
        print(f"CPU: {cpu:.1f}%  |  RAM: {bytes_to_human(mem)}")


stop_event = threading.Event()
t = threading.Thread(target=monitor, daemon=True)
t.start()

# ... your code here ...
# or just "sleep"
time.sleep(3)  # example: simulate work for 3 seconds

stop_event.set()

CPU: 0.3%  |  RAM: 85.33 MiB
CPU: 0.2%  |  RAM: 85.38 MiB
CPU: 0.2%  |  RAM: 85.38 MiB
CPU: 0.2%  |  RAM: 85.38 MiB
CPU: 0.4%  |  RAM: 85.38 MiB


CPU: 1.1%  |  RAM: 85.38 MiB


### 3.2) Continuous Monitoring Example: `tracemalloc`

In [9]:
from collections import deque
import tracemalloc


def track_queue_memory(n_items=12000, report_every=3000):
    user_records_queue = deque()
    checkpoints = []

    tracemalloc.start()

    # Build data in memory (deque + nested data types)
    for i in range(n_items):
        record = {
            "id": i,
            "name": f"user_{i}",
            "values": [j * j % 101 for j in range(20)],
        }
        user_records_queue.append(record)

        if (i + 1) % report_every == 0:
            current, peak = tracemalloc.get_traced_memory()
            checkpoints.append((i + 1, current, peak))

    # Simulate processing
    checksum = 0
    while user_records_queue:
        item = user_records_queue.popleft()
        checksum += sum(item["values"])

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    print("Checkpoints (items, current MB, peak MB):")
    for count, cur, pk in checkpoints:
        print(f"{count:>6} -> {bytes_to_human(cur)} | peak {bytes_to_human(pk)}")
    print(f"Final current: {bytes_to_human(current)}")
    print(f"Peak memory  : {bytes_to_human(peak)}")
    print(f"Checksum: {checksum}")


track_queue_memory(n_items=20000)

Checkpoints (items, current MB, peak MB):
  3000 -> 1.48 MiB | peak 1.48 MiB
  6000 -> 2.97 MiB | peak 2.97 MiB
  9000 -> 4.46 MiB | peak 4.46 MiB
 12000 -> 5.96 MiB | peak 5.96 MiB
 15000 -> 7.45 MiB | peak 7.45 MiB
 18000 -> 8.95 MiB | peak 8.95 MiB
Final current: 20.29 KiB
Peak memory  : 9.95 MiB
Checksum: 17080000


## 4) From the (UX-based) terminal *(no code changes)*

```bash
# One-off snapshot
top -pid $(pgrep -f your_script.py) -l 1

# Or use the `time` command for total CPU time + peak RAM
/usr/bin/time -l python your_script.py
```